# Chapter 2 — Knowledge Graphs and the Geometric Memory Substrate

*A fact is a triple. A knowledge graph is typed facts. GMS places them in a space where a triple's plausibility is a distance.*

In [ ]:
import sys, torch
if not torch.cuda.is_available():
    print(
        "This notebook requires a GPU (CUDA) — no GPU detected.\n"
        "Please re-run on a machine with a CUDA-capable GPU."
    )
    sys.exit(0)

## Objective

Build a small knowledge-graph store from the bank's policy, query it (including a multi-hop chain), and see the one thing GMS adds over a plain graph: a *distance* that says how well a triple fits what the store holds. This is the substrate the rest of the book runs on; here it is built and inspected in isolation, on a store separate from the agent's.

## The source: policy as a document

The store is built from a small excerpt of the bank policy — a fee schedule, the workflow authorizations, the regulatory flags and the reversal authority.

In [ ]:
from pathlib import Path
ROOT = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code')) if (c / 'data').exists()), Path('.'))
DOC = ROOT / 'data' / 'ch2_bank_policy.md'
print(DOC.read_text()[:520])

## From document to knowledge graph

The GEODE build parses the tables into typed triples and trains the geometric store. It is built once and loaded thereafter, so the numbers below are reproducible.

In [ ]:
import torch
from knowlytix.knowledge.geode import build_rag_store, make_default_trainer
from knowlytix.core.config import GeometryConfig, TrainConfig, CapLossConfig
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

STORE = ROOT / 'data' / 'gms_ch2_store'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = DocGMSConfig(store_path=str(STORE), ingest_mode='regex', loss_mode='cap',
                   geometry=GeometryConfig(d_v=64, d_u=64, m=32, d=32),
                   cap=CapLossConfig(n_boundary=4),
                   train=TrainConfig(epochs=400, batch_size=64, neg_samples=8,
                                     lr=5e-3, lr_riemannian=2e-3))
if STORE.exists():                      # built once; loaded on every run after
    store = GMSExpertStore(cfg, device=device); store.load()
else:
    res = build_rag_store(str(DOC), cfg, device=device,
                          geode_trainer=make_default_trainer(device, epochs=120),
                          max_iters=3)
    store = res.store
print('entities:', len(store.adapter.entity_to_idx),
      ' relations:', len(store.adapter.relation_to_idx),
      ' triples:', len(store.query_triples()))

## A fact is a triple `(head, relation, tail)`

A knowledge graph stores typed facts, not strings. Each row of each policy table became triples under a named relation.

In [ ]:
from collections import defaultdict
by_rel = defaultdict(list)
for h, r, t in sorted(store.query_triples()):
    by_rel[r].append((h, t))
for r in ('has_fee_amount', 'has_enables', 'has_applies_when', 'has_escalates_to'):
    print(r)
    for h, t in by_rel[r]:
        print(f'    {h} -> {t}')

## Querying multi-hop: the regulatory escalation chain

The evidence in a complaint implicates a regulatory flag, and the flag routes to an escalation target. That is two hops across *different* relations — the reasoning `flag_regulatory` performs when it walks the graph.

In [ ]:
def heads_with(relation, tail):
    return [h for h, r, t in store.query_triples(relation=relation, tail=tail)]
def tails_of(head, relation):
    return [t for h, r, t in store.query_triples(head=head, relation=relation)]

evidence = 'unfair_or_abusive_fee'
for flag in heads_with('has_applies_when', evidence):        # hop 1: evidence -> flag
    target = tails_of(flag, 'has_escalates_to')              # hop 2: flag -> escalation
    thr = tails_of(flag, 'has_threshold')
    print(f'{evidence} -> {flag} -> escalate to {target} (threshold {thr})')

## What GMS adds: plausibility as a distance

A plain graph answers only *is this triple stored?* GMS answers *how well does this triple fit?* — a small distance for a fact that belongs, a large one for a fabricated or out-of-place one. The same primitive scores a fact against what the policy holds and a workflow transition against what the workflow permits.

In [ ]:
print('fact check  (has_fee_amount):')
for tail, label in [('35.0', 'committed'), ('45.0', 'wrong (wire fee)')]:
    s = store.score_triple('overdraft', 'has_fee_amount', tail)
    print(f'  (overdraft, has_fee_amount, {tail}) -> {s:.3f}  {label}')
print('transition check (has_enables):')
for tail, label in [('extract', 'legal next step'), ('draft_response', 'skips ahead')]:
    s = store.score_triple('classify', 'has_enables', tail)
    print(f'  (classify, has_enables, {tail}) -> {s:.3f}  {label}')

## Exact numbers, recalled losslessly

Numbers are not scored; they are read back exactly from Exact Numerical Memory, with no model in the path.

In [ ]:
from forgeloop.agents.gms_backend import GMSMemory
mem = GMSMemory(store)
for cat, eid in [('fee_schedule', 'overdraft/per_occurrence'),
                 ('fee_schedule', 'wire_international/per_transaction'),
                 ('reversal_authority', 'representative')]:
    print(f'  {cat}/{eid} = {mem.lookup_enm(cat, eid)}')

## The operator set

Plausibility is one of a small fixed set of operators. `query_triples` and `lookup_enm` are exact reads of the asserted graph; the rest read the trained geometry. `link_predict` ranks the plausible tails for a gap (a prediction, never an assertion), and `fuzzy_match_entity` resolves a surface phrase to a canonical node. `tension_energy` (contradiction) and `check_holonomy` (path composition) need structure this small policy store does not contain, and are demonstrated in the chapters that have it (Chapters 9 and 11).

In [ ]:
# link_predict ranks the plausible tails for a gap; the committed value ranks first
print('link_predict(overdraft, has_fee_amount):')
for tail, dist in store.link_predict('overdraft', 'has_fee_amount', top_k=4):
    print(f'  {tail:>5}  {dist:.3f}')

# fuzzy_match_entity resolves a surface phrase to a canonical node
print('fuzzy_match_entity:')
for q in ['overdraft fee', 'escalate']:
    print(f'  {q!r} -> {store.fuzzy_match_entity(q)!r}')

## Visualize the graph on the sphere

GMS places every entity on a sphere and every relation as an operator, so a triple's plausibility is the geodesic distance the earlier cell printed. The figure renders the entities and the triples that connect them.

In [ ]:
import sys
for up in (Path.cwd(), Path.cwd().parent, Path.cwd().parent / 'notebooks',
           Path.cwd() / 'notebooks'):
    if (up / 'kg_sphere.py').exists():
        sys.path.insert(0, str(up)); break
import kg_sphere as K
kg = K.load_gms_store(str(STORE), source='v')
print(f'{len(kg.labels)} entities, {len(kg.triples)} triples')
fig = K.visualize(kg, show=False)
fig

## Reload check

The store is an artifact on disk; loading it returns the same triples, which is what makes a stored verdict replayable.

In [ ]:
reloaded = GMSExpertStore(cfg, device=device)
assert reloaded.load(), 'store failed to load'
assert sorted(reloaded.query_triples()) == sorted(store.query_triples())
print('reload OK:', len(reloaded.query_triples()), 'triples')

## Summary

A knowledge graph stores typed facts as triples. GMS adds a geometry in which a triple's plausibility is a distance, which the book uses to verify a claim (Chapter 3), gate a tool call (Chapter 6), approve a plan (Chapter 8), hold memory (Chapter 9) and govern the runtime (Chapter 12). The internals of the geometry and the calibration of each threshold are the subject of Appendix C. Distance is the central one of a small fixed set of operators (`score_triple`, `link_predict`, `tension_energy`, `check_holonomy`, `fuzzy_match_entity`) alongside the two exact reads (`query_triples`, `lookup_enm`); every later use of the substrate calls one of them.


In [ ]:
# Self-check
assert len(store.query_triples()) > 0
# multi-hop resolves the escalation chain
assert 'compliance' in [t for f in heads_with('has_applies_when', 'unfair_or_abusive_fee')
                        for t in tails_of(f, 'has_escalates_to')]
# a distance separates a committed fact from a wrong one, and a legal step from a skip
assert store.score_triple('overdraft', 'has_fee_amount', '35.0') < \
       store.score_triple('overdraft', 'has_fee_amount', '45.0')
assert store.score_triple('classify', 'has_enables', 'extract') < \
       store.score_triple('classify', 'has_enables', 'draft_response')
assert mem.lookup_enm('fee_schedule', 'overdraft/per_occurrence') == 35.0
# link_predict ranks the committed fee first, and fuzzy match resolves a surface phrase
assert store.link_predict('overdraft', 'has_fee_amount', top_k=4)[0][0] == '35.0'
assert store.fuzzy_match_entity('overdraft fee') == 'overdraft'
print('OK')